<a href="https://colab.research.google.com/github/jadenvv/selfAttentionSentiment/blob/main/selfAttentionSentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [131]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
from google.colab import userdata
os.environ['KAGGLE_KEY']= userdata.get("Kaggle_key")
os.environ['KAGGLE_USERNAME']= userdata.get("Kaggle_user")
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset,DataLoader

In [101]:

training_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,"jp797498e/twitter-entity-sentiment-analysis", "twitter_training.csv")
training_df.columns = ["ID","TOPIC",'SENTIMENT', "TWEET"]
training_df.drop_duplicates(subset="TWEET", inplace=True,  ignore_index=True)
training_df.dropna(subset=["TWEET", "SENTIMENT"], inplace=True, ignore_index=True)
sent,reverse_indices  = np.unique(training_df["SENTIMENT"], return_inverse=True)
training_df.SENTIMENT = reverse_indices
training_df['TWEET'] = [x+ " <EOS>"for x in training_df["TWEET"]]
vocab = list(set((" ".join(training_df["TWEET"].tolist())).split(' ')))
stoi = {text:idx for idx,text in enumerate(vocab)}
itos = {idx:text for idx,text in enumerate(vocab)}


Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.


## Getting the context size
decided on a context size of 64 since the way I create the vocab is based on splitting on whitespace so granted we are gonna get

Addionally,  99 percent of the dataset has lenghts less than 64

In [154]:
np.percentile(n, 99)

np.float64(60.0)

In [152]:
n = np.array([len(x.split(' ')) for x in training_df.TWEET])
over = training_df.loc[n > 64 ]
len(over.TWEET.iloc[1].split(' '))


66

In [119]:
class SentDataset(Dataset):
  def __init__(self,df):
      self.tweet_data= [
          np.array([stoi[x] for x in tweet.split(' ')])
          for tweet in df.TWEET
      ]
      self.sentiment = df["SENTIMENT"]
  def __getitem__(self,idx):
      tweet = self.tweet_data[idx]
      sentiment = self.sentiment.iloc[idx]
      return tweet
  def __len__(self):
      return len(self.tweet_data)

In [120]:
sentdata  = SentDataset(training_df)


In [121]:
dataLoaderSent=DataLoader(sentdata,64, shuffle=True, num_workers=8,drop_last=True)

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 155, in collate
    return collate_fn_map[elem_type](batch, collate_fn_map=collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/